# Corporate Reorganization — ModernBERT retriever training (SageMaker)

This notebook launches SageMaker training jobs using `corporate_reorganization/modernbert/train_sm.py`.

Run sequence:
1. Rebuild `../data/final_annotations_gold/processed/` so it contains the new flat query fields.
2. Run the S3 upload cell.
3. Run the structured training cell if you want a fresh structured model.
4. Run the flat training cell to train the flat-query baseline.
5. Copy the two model artifact URIs into the evaluation notebook.

If you want to reuse an existing structured model artifact, skip the structured training cell and keep its existing S3 URI for evaluation.

In [1]:
import os
import sys

print("Has var:", "SAGEMAKER_EXECUTION_ROLE_ARN" in os.environ)
print("CWD:", os.getcwd())
print("Python:", sys.executable)
print("SAGEMAKER_EXECUTION_ROLE_ARN:", os.getenv("SAGEMAKER_EXECUTION_ROLE_ARN"))


Has var: False
CWD: /home/lbrenap/Documents/projects/legalpaca/corporate_reorganization/notebooks
Python: /home/lbrenap/miniconda3/envs/legalpacaenv/bin/python
SAGEMAKER_EXECUTION_ROLE_ARN: None


In [4]:
from pathlib import Path

import sagemaker
from dotenv import find_dotenv, load_dotenv
from sagemaker.huggingface import HuggingFace

load_dotenv(find_dotenv(usecwd=True))

print("Has var:", "SAGEMAKER_EXECUTION_ROLE_ARN" in os.environ)
role = os.environ["SAGEMAKER_EXECUTION_ROLE_ARN"]
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = "corporate_reorganization/retriever"

print("role:", role)
print("bucket:", bucket)
print("prefix:", prefix)


Has var: True
role: arn:aws:iam::371087393859:role/defaultrole
bucket: sagemaker-us-east-1-371087393859
prefix: corporate_reorganization/retriever


In [6]:
processed_dir = Path("../data/final_annotations_gold/processed").resolve()
assert processed_dir.exists(), f"Missing processed_dir: {processed_dir}"

data_key_prefix = f"{prefix}/data/processed_ablation_v2"

data_s3_uri = session.upload_data(
    path=str(processed_dir),
    bucket=bucket,
    key_prefix=data_key_prefix,
)

inputs = {"data": data_s3_uri}
print("data_s3_uri:", data_s3_uri)
print("Re-run this cell after rebuilding processed/ to overwrite the same S3 keys.")


data_s3_uri: s3://sagemaker-us-east-1-371087393859/corporate_reorganization/retriever/data/processed_ablation_v2
Re-run this cell after rebuilding processed/ to overwrite the same S3 keys.


In [7]:
metric_definitions = [
    {"Name": "eval_loss", "Regex": r"SM_METRIC eval_loss=([0-9eE\.\-]+)"},
    {"Name": "eval_set_recall_at_20", "Regex": r"SM_METRIC eval_set_recall_at_20=([0-9eE\.\-]+)"},
    {"Name": "eval_recall_at_1", "Regex": r"SM_METRIC eval_recall_at_1=([0-9eE\.\-]+)"},
    {"Name": "eval_recall_at_5", "Regex": r"SM_METRIC eval_recall_at_5=([0-9eE\.\-]+)"},
    {"Name": "eval_recall_at_10", "Regex": r"SM_METRIC eval_recall_at_10=([0-9eE\.\-]+)"},
    {"Name": "eval_recall_at_20", "Regex": r"SM_METRIC eval_recall_at_20=([0-9eE\.\-]+)"},
    {"Name": "eval_mrr", "Regex": r"SM_METRIC eval_mrr=([0-9eE\.\-]+)"},
    {"Name": "eval_retrieval_loss", "Regex": r"SM_METRIC eval_retrieval_loss=([0-9eE\.\-]+)"},
    {"Name": "eval_avg_candidates", "Regex": r"SM_METRIC eval_avg_candidates=([0-9eE\.\-]+)"},
]

metric_definitions


[{'Name': 'eval_loss', 'Regex': 'SM_METRIC eval_loss=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_set_recall_at_20',
  'Regex': 'SM_METRIC eval_set_recall_at_20=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_recall_at_1',
  'Regex': 'SM_METRIC eval_recall_at_1=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_recall_at_5',
  'Regex': 'SM_METRIC eval_recall_at_5=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_recall_at_10',
  'Regex': 'SM_METRIC eval_recall_at_10=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_recall_at_20',
  'Regex': 'SM_METRIC eval_recall_at_20=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_mrr', 'Regex': 'SM_METRIC eval_mrr=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_retrieval_loss',
  'Regex': 'SM_METRIC eval_retrieval_loss=([0-9eE\\.\\-]+)'},
 {'Name': 'eval_avg_candidates',
  'Regex': 'SM_METRIC eval_avg_candidates=([0-9eE\\.\\-]+)'}]

In [13]:
base_hyperparameters = {
    "deepspeed": "ds_zero3.json",
    "epochs": 20,
    "learning_rate": 1e-5,
    "temperature": 0.07,
    "batch_size_queries": 4,
    "max_len_query": 4096,
    "max_len_passage": 500,
    "max_pos_per_query": 4,
    "num_same_case_negatives": 56,
    "num_distractor_negatives": 4,
    "distractor_labels": "Background Facts",
    "eval_query_batch_size": 64,
    "eval_passage_batch_size": 256,
    "gradient_accumulation_steps": 8,
}

def make_estimator(*, query_view: str):
    hyperparameters = {**base_hyperparameters, "query_view": query_view}
    return HuggingFace(
        entry_point="train_sm.py",
        source_dir="../modernbert",
        instance_type="ml.g5.12xlarge",
        instance_count=1,
        role=role,
        transformers_version="4.49.0",
        pytorch_version="2.5.1",
        py_version="py311",
        hyperparameters=hyperparameters,
        metric_definitions=metric_definitions,
        distribution={"mpi": {"enabled": True, "processes_per_host": 4}},
        environment={"PYTORCH_ALLOC_CONF": "expandable_segments:True"},
    )

base_hyperparameters


{'deepspeed': 'ds_zero3.json',
 'epochs': 20,
 'learning_rate': 1e-05,
 'temperature': 0.07,
 'batch_size_queries': 4,
 'max_len_query': 4096,
 'max_len_passage': 500,
 'max_pos_per_query': 4,
 'num_same_case_negatives': 56,
 'num_distractor_negatives': 4,
 'distractor_labels': 'Background Facts',
 'eval_query_batch_size': 64,
 'eval_passage_batch_size': 256,
 'gradient_accumulation_steps': 8}

In [16]:
# Run this cell only if you want to train a fresh structured model.
structured_estimator = make_estimator(query_view="structured")
structured_estimator.fit(inputs, wait=True, logs=True)
structured_model_s3_uri = structured_estimator.model_data
structured_model_s3_uri


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-03-16-00-09-29-909


2026-03-16 00:09:31 Starting - Starting the training job
2026-03-16 00:09:31 Pending - Training job waiting for capacity.........
2026-03-16 00:10:37 Pending - Preparing the instances for training...
2026-03-16 00:11:20 Downloading - Downloading the training image...........................
2026-03-16 00:15:43 Training - Training image download completed. Training in progress...bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.163.01
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-03-16 00:16:20,747 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-03-16 00:16:20,786 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-16 00:16:20,795 sagemaker_pytorch_container.training INFO     Block unt

's3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-16-00-09-29-909/output/model.tar.gz'

In [14]:
# Run this cell to train the flat-query baseline.
flat_estimator = make_estimator(query_view="flat_masked")
flat_estimator.fit(inputs, wait=True, logs=True)
flat_model_s3_uri = flat_estimator.model_data
flat_model_s3_uri


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-03-15-23-12-42-656


2026-03-15 23:12:44 Starting - Starting the training job
2026-03-15 23:12:44 Pending - Training job waiting for capacity......
2026-03-15 23:13:20 Pending - Preparing the instances for training...
2026-03-15 23:14:04 Downloading - Downloading the training image........................
2026-03-15 23:18:01 Training - Training image download completed. Training in progress....bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 550.163.01
Current installed NVIDIA driver version is 570.211.01
Skipping CUDA compat setup as newer NVIDIA driver is installed
2026-03-15 23:18:40,981 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-03-15 23:18:41,019 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-03-15 23:18:41,028 sagemaker_pytorch_container.training INFO     Block until al

's3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-15-23-12-42-656/output/model.tar.gz'

In [17]:
print("structured_model_s3_uri:", globals().get("structured_model_s3_uri"))
print("flat_model_s3_uri:", globals().get("flat_model_s3_uri"))


structured_model_s3_uri: s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-16-00-09-29-909/output/model.tar.gz
flat_model_s3_uri: s3://sagemaker-us-east-1-371087393859/huggingface-pytorch-training-2026-03-15-23-12-42-656/output/model.tar.gz


## Output artifacts

After each job finishes, the model artifact is in `<estimator>.model_data` as an S3 `model.tar.gz`.

Inside `model.tar.gz` the script writes:
- `model.safetensors`
- `wrapper_config.json`
- tokenizer files
- `encoder_config/`

Use the structured and flat S3 URIs in the evaluation notebook.